# Create windows for CNN training

In [ ]:
import sys
sys.path.append('../../')

import numpy as np
import matplotlib.pyplot as plt

from param_classes.workflows.input_output import InputDataParams
from workflows.run_util.io import collect_paths_from_table

input_params = InputDataParams()

## Data settings

Set raw input dir. You need to give the base path where the group directories can be found.

In [ ]:
input_params.raw_data.path ='E:\\Data\\BogiSejtek\\20251007\\data\\invivo'
input_params.raw_data.extension = 'mat'

Set the base path of the directory where csv files containing the spike features and categories can be found.
> It needs to be the same structure as the `raw_data` directory.

In [ ]:
input_params.spike_features.path = 'E:\\Data\\BogiSejtek\\20251007\\data\\invivo\\Preprocessed\\SpikeFeatures'
input_params.spike_features.extension = 'csv'

## Input-output settings

Set the Excel table containing the cell attributes, grouping and selector information.

In [ ]:
input_selector = {
    'mode': 'table',
    'table_path': 'E:\\Data\\BogiSejtek\\20251007\\data\\invivo\\Preprocessed\\data_invivo_noisetest.xlsx',
    'select_rules': [
        {'selected': 1}
    ]
}

Set the output base directory path. All results of this analysis (cellwise aggregated tables, grouped results, figures, statistics) is going to saved there.

In [ ]:
output_path = 'E:\\Data\\BogiSejtek\\20251007\\data\\invivo\\Preprocessed\\SpikeOcurrencePrediction\\01_training_data'

## Analysis parameters

In [ ]:
signal_window_size = 0.25 # seconds
signal_window_shift = 0.05 # seconds
signal_fs = 1000 # Hz
spike_window_size = 0.250
# spike_types = ['AIS-AP', 'UNKNOWN', 'RA-AP']
spike_types = ['RA-AP']


## Run

### Preparation

In [ ]:
n_window_samples = int(signal_window_size * signal_fs)
n_shift_samples = int(signal_window_shift * signal_fs)

In [ ]:
input_files = collect_paths_from_table(input_params, input_selector)

In [ ]:
for input_file in input_files:
    if input_file.name != 'B18_d3t1c1':
        continue
    data = input_file.data.load()
    spikes = data.spikes
    signal = data.get_signal('wideband', fs=signal_fs)
    times, values = signal.times, signal.values
    signal_segments = []
    spike_ocurrences = []
    for win_start_i in range(0, len(values) - n_window_samples, n_shift_samples):
        win_end_i = win_start_i + n_window_samples
        signal_segment = values[win_start_i:win_end_i]
        signal_segments.append(signal_segment)
        spike_window = tuple([
            times[win_end_i] + t for t in [0, spike_window_size]
        ])
        spikes_in_window = spikes[
            (spike_window[0]<=spikes['timestamp']) &
            (spikes['timestamp']<=spike_window[1])
        ]
        spike_ocurrences.append(
            np.asarray([
                np.any(spikes_in_window['spike_type'] == spike_type).astype(float)
                for spike_type in spike_types
            ])
        )

In [ ]:
X = np.stack(signal_segments, axis=0)
y = np.stack(spike_ocurrences, axis=0)

In [ ]:
from workflows.spike_occurrence_prediction.deep_network_factories import create_cnn_model
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.model_selection import StratifiedKFold

splitter = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)


for fold, (train_index, test_index) in enumerate(splitter.split(X, y[:, -1])):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    model = create_cnn_model(
        layer_params={
            'input': {'type': 'Input', 'shape': (n_window_samples, 1)},
            'conv': [
                {'type': 'Conv1D', 'filters': 16, 'kernel_size': 5, 'activation': 'relu'},
                {'type': 'MaxPooling1D', 'pool_size': 2},
                {'type': 'Conv1D', 'filters': 32, 'kernel_size': 5, 'activation': 'relu'},
                {'type': 'MaxPooling1D', 'pool_size': 2},
            ],
            'flatten': {'type': 'GlobalAveragePooling1D'},
            'fcn': [
                {'type': 'Dense', 'units': 128, 'activation': 'relu'},
                {'type': 'Dropout', 'rate': 0.5},
                {'type': 'Dense', 'units': 64, 'activation': 'relu'},
                {'type': 'Dropout', 'rate': 0.5}
            ],
            'output': {'type': 'Dense', 'units': y.shape[1], 'activation': 'sigmoid'}
        },
        learning_rate=0.001
    )

    model.fit(
        X_train, y_train,
        epochs=100,
        batch_size=32,
        validation_data=(X_test, y_test)
    )
    y_pred = model.predict(X_test)

    for i, spike_type in enumerate(spike_types):
        precision = precision_score(y_test[:, i], y_pred[:, i]>0.5)
        recall = recall_score(y_test[:, i], y_pred[:, i]>0.5)
        f1 = f1_score(y_test[:, i], y_pred[:, i]>0.5)
        print(f'Fold {fold}, {spike_type}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}')